# Experiment A.01 — setup and preflight

In [ ]:
import json,os,subprocess
from pathlib import Path
R=Path.home()/"async-vla-latency-bench"; PY=Path.home()/"venv-stage1-id/bin/python"; P=Path.home()/"LIBERO-plus"; S3=Path.home()/"stage3"; S3B=Path.home()/"stage3b"; S3C=Path.home()/"stage3c"; OUT=Path.home()/"experiment_a"; OUT.mkdir(exist_ok=True)
required=(R,P,P/"libero/libero/assets",PY,S3/"stage3_episode_results.csv",S3B/"stage3b_episode_results.csv",S3C/"stage3c_initialization_audit.json")
missing=[str(p) for p in required if not p.exists()]
if missing: raise SystemExit(f"STOP: missing prerequisites: {missing}")
s3c=json.loads((S3C/"stage3c_initialization_audit.json").read_text()); assert s3c["status"]=="fail" and s3c["reset_operations"]==144 and s3c["validated_initializations"]==0
env=os.environ.copy(); env.update({"MUJOCO_GL":"egl","PYOPENGL_PLATFORM":"egl","MPLBACKEND":"Agg"})
subprocess.run([str(PY),"-m","pytest","-q",str(R/"async_vla_benchmark/tests")],cwd=R,env=env,check=True)
gpu="1"; line=subprocess.run(["nvidia-smi",f"--id={gpu}","--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version","--format=csv,noheader,nounits"],capture_output=True,text=True,check=True).stdout.strip(); print(line)
name,total,used,util,driver=[x.strip() for x in line.split(',')]; assert "A100" in name
if int(used)>=500 or int(util)>=5: raise SystemExit("STOP: select an idle physical A100")
provenance={"repository_sha":"anonymous-source","libero_plus_sha":subprocess.run(["git","-C",str(P),"rev-parse","HEAD"],capture_output=True,text=True,check=True).stdout.strip(),"gpu":line,"stage3c_gate_status":"failed_closed"}
(OUT/"experiment_a_preflight_environment.json").write_text(json.dumps(provenance,indent=2)+"\n"); print("PASS: Experiment A preflight complete; Stage 3C failure acknowledged")